# Sales Analytics Notebook
Reads raw sales data from the **SalesLakehouse** Delta table and writes
aggregated daily revenue metrics back as a Gold layer table.

In [ ]:
# Read raw (Bronze) sales data
df_raw = spark.read.format('delta').load('Tables/bronze_sales')
df_raw.printSchema()

In [ ]:
from pyspark.sql import functions as F

# Silver: clean and standardise
df_silver = (
    df_raw
    .filter(F.col('order_status') != 'CANCELLED')
    .withColumn('order_date', F.to_date('order_timestamp'))
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .select('order_id', 'order_date', 'product_id', 'region', 'revenue')
)
df_silver.cache()

In [ ]:
# Gold: daily revenue by region
df_gold = (
    df_silver
    .groupBy('order_date', 'region')
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.countDistinct('order_id').alias('order_count')
    )
    .orderBy('order_date', 'region')
)

# Write back to Gold layer
df_gold.write.format('delta').mode('overwrite').saveAsTable('gold_daily_revenue')
print('Gold table written successfully.')